# Comparative and Ablation Analysis: Diebold-Mariano Test with HLN Correction

Tests whether the differences in RMSE-based predictive accuracy between models are statistically significant.

| Analysis | Research question | Comparisons |
|---|---|---|
| **Comparative** | Is there a significant difference in RMSE-based predictive accuracy between the final proposed model and each benchmark? | Proposed (Hybrid SARIMAX-LSTM + CSI + PL) vs SARIMAX, LSTM, XGBoost, Climatology and Seasonal naive (5 tests) |
| **Ablation** | Is there a significant difference in RMSE-based predictive accuracy between the individual components of the stacked model? | Hybrid SARIMAX-LSTM vs + CSI · + CSI vs + CSI + PL · Hybrid SARIMAX-LSTM vs + CSI + PL (3 tests) |

## Data: the full out-of-sample record
All forecasts are genuine year-ahead, out-of-sample forecasts made from a December origin:
- **2015–2022:** rolling validation, 8 forecast origins
- **2023–2024:** final testing, 2 forecast origins

Together that is **120 months × 93 grids = 11,160 forecasts per model**.

## Test: Diebold-Mariano with the Harvey-Leybourne-Newbold correction (DM-HLN)
- **Loss:** squared forecast error. Equal mean squared error is equivalent to equal RMSE, so the test compares RMSE-based accuracy directly.
- **Loss differential:** for each month *t*, the difference in squared error is averaged over the 93 grids: `d_t = (1/93) Σ_i [ e²_A,it − e²_B,it ]`. Averaging first absorbs the strong spatial correlation between neighbouring grids, so the 11,160 correlated errors aren't treated as independent evidence. The mean of `d_t` equals `MSE_A − MSE_B`.
- **DM statistic:** `DM = d̄ / √(V̂ / T)`, where `V̂ = γ̂₀ + 2 Σ_{k=1}^{h−1} γ̂_k`. With h = 12, the variance allows for correlation up to 11 months apart, because each year's 12 forecasts come from the same origin (Diebold & Mariano, 1995). If `V̂` is not positive, Bartlett weights are used instead.
- **HLN correction:** `DM* = DM · √[(T + 1 − 2h + h(h − 1)/T) / T]`, compared with Student's t with T − 1 degrees of freedom (Harvey, Leybourne & Newbold, 1997). Here T = 120 months.
- **Hypotheses:** H₀: E[d_t] = 0 (equal RMSE-based accuracy) vs H₁: E[d_t] ≠ 0 (two-sided, matching the research questions). The sign of d̄ shows which model is more accurate.
- **Multiple comparisons:** p-values are Holm-adjusted within each family (Holm, 1979). Conclusions use the adjusted p-value at α = 0.05.

A supplementary section repeats the test for each grid cell, with Benjamini-Hochberg false-discovery-rate control across the 93 grids, for mapping where differences are significant.

## Outputs (`outputs/significance_tests/`)
- `model_accuracy_full_record.csv`: pooled RMSE, MAE and nRMSE for every model, 2015–2024
- `comparative_dm_hln.csv` and `ablation_dm_hln.csv`: one row per test
- `loss_differentials_monthly.csv`: the monthly loss-differential series behind every test
- `per_grid_dm_hln.csv`: per-grid tests (supplementary, for maps)

Run after `Standalone_Models.ipynb`, `Final_Testing_Standalone.ipynb`, `Hybrid_Models.ipynb` and `Final_Testing_Hybrid.ipynb`, from the `Philippine Spatial Map` folder. It takes a few seconds.

In [ ]:
"""
comparative_ablation_dm_hln.py

Purpose:
    Diebold-Mariano tests with the Harvey-Leybourne-Newbold
    small-sample correction for the comparative and ablation
    analyses, over the full out-of-sample record (2015-2024).

Input:
    outputs/standalone_models/rolling_predictions/
    outputs/final_test/predictions/
    outputs/hybrid_models/rolling_predictions/
    outputs/final_test_hybrid/predictions/

Output:
    outputs/significance_tests/
"""

from pathlib import Path

import numpy as np
import pandas as pd
from scipy import stats
from statsmodels.stats.multitest import multipletests


# ============================================================
# 1. USER SETTINGS
# ============================================================

STANDALONE_ROLLING = Path("outputs/standalone_models/rolling_predictions")

STANDALONE_FINAL = Path("outputs/final_test/predictions")

HYBRID_ROLLING = Path("outputs/hybrid_models/rolling_predictions")

HYBRID_FINAL = Path("outputs/final_test_hybrid/predictions")

OUTPUT_DIR = Path(
    "outputs/significance_tests"
)

# Full out-of-sample record
FIRST_YEAR = 2015

LAST_YEAR = 2024

# Forecast horizon: 12 monthly forecasts from each December origin
HORIZON = 12

# Two-sided: is there a significant difference?
ALTERNATIVE = "two-sided"

ALPHA = 0.05

PROPOSED = "Hybrid SARIMAX-LSTM + CSI + PL (proposed)"


# ------------------------------------------------------------
# Forecast files per model: rolling validation + final test
# ------------------------------------------------------------

MODELS = {
    PROPOSED: [
        HYBRID_ROLLING / "hybrid_csi_pl_rolling_predictions.csv",
        HYBRID_FINAL / "hybrid_csi_pl_final_test_predictions.csv"
    ],
    "Hybrid SARIMAX-LSTM + CSI": [
        HYBRID_ROLLING / "hybrid_csi_rolling_predictions.csv",
        HYBRID_FINAL / "hybrid_csi_final_test_predictions.csv"
    ],
    "Hybrid SARIMAX-LSTM": [
        HYBRID_ROLLING / "hybrid_rolling_predictions.csv",
        HYBRID_FINAL / "hybrid_final_test_predictions.csv"
    ],
    "SARIMAX": [
        STANDALONE_ROLLING / "sarimax_rolling_predictions.csv",
        STANDALONE_FINAL / "sarimax_final_test_predictions.csv"
    ],
    "LSTM": [
        STANDALONE_ROLLING / "lstm_rolling_predictions.csv",
        STANDALONE_FINAL / "lstm_final_test_predictions.csv"
    ],
    "XGBoost": [
        STANDALONE_ROLLING / "xgboost_rolling_predictions.csv",
        STANDALONE_FINAL / "xgboost_final_test_predictions.csv"
    ],
    "Climatology": [
        STANDALONE_ROLLING / "climatology_rolling_predictions.csv",
        STANDALONE_FINAL / "climatology_final_test_predictions.csv"
    ],
    "Seasonal naive": [
        STANDALONE_ROLLING / "seasonal_naive_rolling_predictions.csv",
        STANDALONE_FINAL / "seasonal_naive_final_test_predictions.csv"
    ]
}


# ============================================================
# 2. CREATE OUTPUT DIRECTORY
# ============================================================

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)


# ============================================================
# 3. LOAD FORECASTS AND CHECK COMPLETENESS
# ============================================================

KEY = ["grid_number", "target_date"]

errors = {}

actuals = {}

grid_info = None

for model, paths in MODELS.items():

    missing = [str(path) for path in paths if not path.exists()]

    if missing:
        raise FileNotFoundError(f"{model}: missing {missing}. Run the model notebooks first.")

    frame = pd.concat(
        [pd.read_csv(path, parse_dates=["target_date"]) for path in paths],
        ignore_index=True
    )

    frame = frame[frame["target_date"].dt.year.between(FIRST_YEAR, LAST_YEAR)]

    if frame.duplicated(KEY).any():
        raise ValueError(f"{model}: duplicate grid-month forecasts")

    frame = frame.set_index(KEY).sort_index()

    errors[model] = frame["predicted"] - frame["actual"]

    actuals[model] = frame["actual"]

    if grid_info is None:
        grid_info = (
            frame.reset_index()[["grid_number", "grid_id", "latitude", "longitude"]]
            .drop_duplicates("grid_number")
            .set_index("grid_number")
        )


errors = pd.DataFrame(errors)

actuals = pd.DataFrame(actuals)

n_grids = errors.index.get_level_values("grid_number").nunique()

n_months = errors.index.get_level_values("target_date").nunique()

expected = n_grids * n_months

if errors.isna().any().any() or len(errors) != expected:
    raise ValueError("Models do not cover the same grid-months")

if (actuals.max(axis=1) - actuals.min(axis=1)).max() > 1e-9:
    raise ValueError("Observed GHI differs between forecast files")

actual = actuals.iloc[:, 0]

print(
    f"Full record: {FIRST_YEAR}-{LAST_YEAR}   grids: {n_grids}   months: {n_months}   "
    f"forecasts per model: {len(errors)}   models: {len(MODELS)}"
)


# ============================================================
# 4. POOLED ACCURACY OVER THE FULL RECORD
# ============================================================

accuracy = pd.DataFrame({
    "RMSE":        np.sqrt((errors ** 2).mean()),
    "MAE":         errors.abs().mean(),
    "nRMSE_pct":   np.sqrt((errors ** 2).mean()) / actual.mean() * 100,
    "n_forecasts": errors.count()
})

accuracy.index.name = "model"

accuracy = accuracy.sort_values("RMSE")

accuracy.to_csv(OUTPUT_DIR / "model_accuracy_full_record.csv")

print("\nPooled accuracy, full record (kWh/m²/day):")

print(accuracy.round(4).to_string())

## DM-HLN test function

`dm_hln_test(d)` takes the monthly loss-differential series `d_t = mean_i(e²_A − e²_B)` and returns the HLN-corrected statistic and its p-value. **A negative d̄ (and statistic) means model A is more accurate.**

In [ ]:
# ============================================================
# 5. DIEBOLD-MARIANO TEST WITH HLN CORRECTION
# ============================================================

def dm_hln_test(d, h=HORIZON, alternative=ALTERNATIVE):
    """
    Diebold-Mariano test (Diebold & Mariano, 1995) with the
    Harvey-Leybourne-Newbold (1997) small-sample correction.

    d : loss differential series, L_A - L_B (negative = A more accurate)
    h : forecast horizon; autocovariances up to lag h-1 are used
    """

    d = np.asarray(d, dtype=float)

    T = len(d)

    d_bar = d.mean()

    # Autocovariances gamma_0 ... gamma_{h-1}
    gamma = np.array([
        np.sum((d[k:] - d_bar) * (d[:T - k] - d_bar)) / T
        for k in range(h)
    ])

    long_run_variance = gamma[0] + 2 * gamma[1:].sum()

    kernel = "rectangular"

    if long_run_variance <= 0:

        # Bartlett weights guarantee a positive variance
        weights = 1 - np.arange(1, h) / h

        long_run_variance = gamma[0] + 2 * (weights * gamma[1:]).sum()

        kernel = "Bartlett"

    dm = d_bar / np.sqrt(long_run_variance / T)

    correction = np.sqrt((T + 1 - 2 * h + h * (h - 1) / T) / T)

    statistic = correction * dm

    df = T - 1

    if alternative == "two-sided":
        p_value = 2 * stats.t.sf(abs(statistic), df)
    elif alternative == "less":
        p_value = stats.t.cdf(statistic, df)
    else:
        p_value = stats.t.sf(statistic, df)

    return {
        "T": T,
        "mean_loss_differential": d_bar,
        "DM": dm,
        "DM_HLN": statistic,
        "df": df,
        "p_value": p_value,
        "variance_kernel": kernel
    }


def loss_differential(model_a, model_b):
    """Squared-error differential per grid-month: e²_A - e²_B."""

    return errors[model_a] ** 2 - errors[model_b] ** 2


def run_family(pairs, family):
    """
    DM-HLN test for every (A, B) pair, with Holm-adjusted p-values
    within the family. Returns (results table, monthly differentials).
    """

    rows, series = [], {}

    for model_a, model_b in pairs:

        diff = loss_differential(model_a, model_b)

        monthly = diff.groupby(level="target_date").mean().sort_index()

        series[f"{model_a} vs {model_b}"] = monthly

        test = dm_hln_test(monthly.to_numpy())

        rmse_a = accuracy.loc[model_a, "RMSE"]

        rmse_b = accuracy.loc[model_b, "RMSE"]

        rows.append({
            "family":      family,
            "model_A":     model_a,
            "model_B":     model_b,
            "RMSE_A":      rmse_a,
            "RMSE_B":      rmse_b,
            "RMSE_diff":   rmse_a - rmse_b,
            "RMSE_diff_pct": (rmse_a - rmse_b) / rmse_b * 100,
            **test
        })

    table = pd.DataFrame(rows)

    table["p_holm"] = multipletests(table["p_value"], method="holm")[1]

    table["significant"] = table["p_holm"] < ALPHA

    table["more_accurate"] = np.where(table["mean_loss_differential"] < 0, table["model_A"], table["model_B"])

    table["conclusion"] = np.where(
        table["significant"],
        "Significant difference; " + table["more_accurate"] + " more accurate",
        "No significant difference"
    )

    return table, pd.DataFrame(series)


def report(table):

    for row in table.itertuples(index=False):

        print(
            f"\n{row.model_A}  vs  {row.model_B}\n"
            f"  RMSE {row.RMSE_A:.4f} vs {row.RMSE_B:.4f}  ({row.RMSE_diff_pct:+.2f}%)\n"
            f"  DM-HLN = {row.DM_HLN:+.3f}  (df = {row.df})   p = {row.p_value:.4f}   "
            f"Holm p = {row.p_holm:.4f}\n"
            f"  -> {row.conclusion}"
        )


print(
    f"DM-HLN settings: T = {n_months} months, h = {HORIZON}, alternative = {ALTERNATIVE}, "
    f"alpha = {ALPHA}, Holm correction within each family"
)

## 1. Comparative analysis: proposed model vs benchmarks

Model A is the proposed model and model B is each benchmark. A negative DM-HLN statistic means the proposed model has the lower RMSE.

In [ ]:
# ============================================================
# 6. COMPARATIVE ANALYSIS
# ============================================================

COMPARATIVE_PAIRS = [
    (PROPOSED, "SARIMAX"),
    (PROPOSED, "LSTM"),
    (PROPOSED, "XGBoost"),
    (PROPOSED, "Climatology"),
    (PROPOSED, "Seasonal naive")
]

comparative, comparative_series = run_family(COMPARATIVE_PAIRS, "comparative")

comparative.to_csv(OUTPUT_DIR / "comparative_dm_hln.csv", index=False)

print("=" * 65)
print(f"COMPARATIVE ANALYSIS ({FIRST_YEAR}-{LAST_YEAR}, DM-HLN, two-sided, Holm-adjusted)")
print("=" * 65)

report(comparative)

print("\nSummary:")

print(
    comparative[["model_B", "RMSE_A", "RMSE_B", "RMSE_diff_pct", "DM_HLN", "p_value", "p_holm", "significant"]]
    .rename(columns={"model_B": "proposed vs"})
    .round(4)
    .to_string(index=False)
)

## 2. Ablation analysis: components of the stacked model

The three components are compared pairwise. In every pair, **model A has more components** than model B, so a negative statistic means adding the component(s) lowered the RMSE.

| Pair | Effect tested |
|---|---|
| Hybrid + CSI vs Hybrid SARIMAX-LSTM | adding the Ineichen-Perez clear-sky index |
| Hybrid + CSI + PL vs Hybrid + CSI | adding the penalized loss function |
| Hybrid + CSI + PL vs Hybrid SARIMAX-LSTM | adding both (the full stack) |

In [ ]:
# ============================================================
# 7. ABLATION ANALYSIS
# ============================================================

ABLATION_PAIRS = [
    ("Hybrid SARIMAX-LSTM + CSI", "Hybrid SARIMAX-LSTM"),
    (PROPOSED, "Hybrid SARIMAX-LSTM + CSI"),
    (PROPOSED, "Hybrid SARIMAX-LSTM")
]

ablation, ablation_series = run_family(ABLATION_PAIRS, "ablation")

ablation.to_csv(OUTPUT_DIR / "ablation_dm_hln.csv", index=False)

pd.concat([comparative_series, ablation_series], axis=1).to_csv(
    OUTPUT_DIR / "loss_differentials_monthly.csv",
    index_label="target_date"
)

print("=" * 65)
print(f"ABLATION ANALYSIS ({FIRST_YEAR}-{LAST_YEAR}, DM-HLN, two-sided, Holm-adjusted)")
print("=" * 65)

report(ablation)

print("\nSummary:")

print(
    ablation[["model_A", "model_B", "RMSE_A", "RMSE_B", "RMSE_diff_pct", "DM_HLN", "p_value", "p_holm", "significant"]]
    .round(4)
    .to_string(index=False)
)

## 3. Supplementary: per-grid tests (for significance maps)

This repeats the DM-HLN test for **each grid cell separately**, using that grid's 120 monthly squared-error differentials. With 93 tests per comparison, about 5 would be "significant" at 5 % by chance alone. The p-values are therefore adjusted across grids with the Benjamini-Hochberg false-discovery-rate procedure at q = 0.05 (Benjamini & Hochberg, 1995; Wilks, 2016).

The results can be overlaid on the RMSE maps, for example by marking grids where the proposed model is significantly more accurate. Treat them as supplementary to the pooled tests above, which answer the research questions.

In [ ]:
# ============================================================
# 8. SUPPLEMENTARY: PER-GRID DM-HLN TESTS WITH FDR CONTROL
# ============================================================

FDR_Q = 0.05

rows = []

for model_a, model_b in COMPARATIVE_PAIRS + ABLATION_PAIRS:

    diff = loss_differential(model_a, model_b)

    pair_rows = []

    for grid_number, series in diff.groupby(level="grid_number"):

        test = dm_hln_test(series.sort_index().to_numpy())

        squared_a = errors.loc[grid_number, model_a] ** 2

        squared_b = errors.loc[grid_number, model_b] ** 2

        pair_rows.append({
            "model_A":     model_a,
            "model_B":     model_b,
            "grid_number": grid_number,
            "RMSE_A":      np.sqrt(squared_a.mean()),
            "RMSE_B":      np.sqrt(squared_b.mean()),
            **test
        })

    pair = pd.DataFrame(pair_rows)

    pair["p_fdr"] = multipletests(pair["p_value"], method="fdr_bh")[1]

    rows.append(pair)


per_grid = pd.concat(rows, ignore_index=True).merge(
    grid_info.reset_index(), on="grid_number", how="left"
)

per_grid["significant_fdr"] = per_grid["p_fdr"] < FDR_Q

per_grid["more_accurate"] = np.where(per_grid["mean_loss_differential"] < 0, "A", "B")

per_grid = per_grid[
    ["model_A", "model_B", "grid_number", "grid_id", "latitude", "longitude",
     "RMSE_A", "RMSE_B", "mean_loss_differential", "DM_HLN", "p_value", "p_fdr",
     "significant_fdr", "more_accurate", "variance_kernel"]
]

per_grid.to_csv(OUTPUT_DIR / "per_grid_dm_hln.csv", index=False)

summary = (
    per_grid.assign(
        A_better_significant=per_grid["significant_fdr"] & (per_grid["more_accurate"] == "A"),
        B_better_significant=per_grid["significant_fdr"] & (per_grid["more_accurate"] == "B"),
        A_lower_RMSE=per_grid["more_accurate"] == "A"
    )
    .groupby(["model_A", "model_B"], sort=False)
    .agg(
        grids=("grid_number", "size"),
        A_lower_RMSE=("A_lower_RMSE", "sum"),
        A_significantly_better=("A_better_significant", "sum"),
        B_significantly_better=("B_better_significant", "sum")
    )
    .reset_index()
)

print("=" * 65)
print(f"PER-GRID DM-HLN TESTS (FDR q = {FDR_Q})")
print("=" * 65)

print(summary.to_string(index=False))

print(f"\nSaved: {OUTPUT_DIR / 'per_grid_dm_hln.csv'}")
print(f"Saved: {OUTPUT_DIR / 'comparative_dm_hln.csv'}")
print(f"Saved: {OUTPUT_DIR / 'ablation_dm_hln.csv'}")
print(f"Saved: {OUTPUT_DIR / 'loss_differentials_monthly.csv'}")
print(f"Saved: {OUTPUT_DIR / 'model_accuracy_full_record.csv'}")

## 4. Figure: RMSE difference maps with significance markers

For each benchmark, every grid cell is coloured by **ΔRMSE = RMSE of the proposed model − RMSE of the benchmark** over its 120 months (2015–2024). This is the same period as the significance tests, so the colours and the markers describe the same forecasts.

- **Blue:** the proposed model is more accurate.
- **Red:** the benchmark is more accurate.
- **Gray:** no difference.
- **Dot:** the difference is significant in the per-grid DM-HLN test after FDR control (q < 0.05).

The colour scale runs from −0.08 to +0.08 kWh/m²/day and is the same in every panel. Larger differences take the end colour. This only happens against seasonal naive, where the proposed model is better by 0.08–0.20 in every grid.

The arms of the blue–red scale are matched in lightness step by step, so equal gains and losses look equally strong.

**Output:** `outputs/figures/rmse_difference_maps_full_record.png` / `.svg` / `.pdf`

In [ ]:
# ============================================================
# 9. FIGURE: RMSE DIFFERENCE MAPS WITH SIGNIFICANCE (FULL RECORD)
# ============================================================

import geopandas as gpd
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap, TwoSlopeNorm
from matplotlib.cm import ScalarMappable


# ------------------------------------------------------------
# Settings
# ------------------------------------------------------------

GRID_FILE = Path("data/processed/philippines_grid.geojson")

BOUNDARY_FILE = Path("data/boundaries/Philippines.shp/PH_Adm0_Country.shp.shp")

FIGURE_DIR = Path("outputs/figures")

FIGURE_DIR.mkdir(parents=True, exist_ok=True)

# Colour scale limit (kWh/m²/day); larger differences take the end colour
LIMIT = 0.08

UNITS_LABEL = "kWh/m²/day"

INK, INK_2, AXIS = "#0b0b0b", "#52514e", "#c3c2b7"

# Diverging scale: blue = proposed more accurate, gray = no difference,
# red = benchmark more accurate (arms matched in lightness step by step)
DIVERGING = ["#0d366b", "#184f95", "#2a78d6", "#6da7ec", "#b7d3f6", "#f0efec",
             "#f6c1c1", "#eb7f7e", "#cc4141", "#892c2b", "#5f1f1e"]

CMAP = LinearSegmentedColormap.from_list("rmse_difference", DIVERGING)

NORM = TwoSlopeNorm(vmin=-LIMIT, vcenter=0.0, vmax=LIMIT)

plt.rcParams.update({
    "font.family":      "Arial",
    "font.size":        7.5,
    "xtick.color":      "#898781",
    "ytick.color":      "#898781",
    "xtick.labelcolor": INK_2,
    "ytick.labelcolor": INK_2,
    "xtick.major.size": 2.5,
    "ytick.major.size": 2.5,
    "xtick.major.width": 0.6,
    "ytick.major.width": 0.6,
    "svg.fonttype":     "none",
    "pdf.fonttype":     42
})


# ------------------------------------------------------------
# Per-grid RMSE differences: proposed minus each benchmark
# ------------------------------------------------------------

BENCHMARKS = [benchmark for _, benchmark in COMPARATIVE_PAIRS]

differences = pd.read_csv(OUTPUT_DIR / "per_grid_dm_hln.csv")

differences = differences[
    (differences["model_A"] == PROPOSED) & differences["model_B"].isin(BENCHMARKS)
].copy()

differences["RMSE_difference"] = differences["RMSE_A"] - differences["RMSE_B"]

grid = gpd.read_file(GRID_FILE)

# Cell centres (stored in the grid file) for the significance markers
grid["x"] = grid["cell_center_longitude"]

grid["y"] = grid["cell_center_latitude"]

boundary = gpd.read_file(BOUNDARY_FILE).to_crs(grid.crs)

boundary["geometry"] = boundary.geometry.simplify(0.005)

LON_LIMITS, LAT_LIMITS = (115.8, 127.2), (3.8, 22.2)

MAP_ASPECT = (LAT_LIMITS[1] - LAT_LIMITS[0]) / (LON_LIMITS[1] - LON_LIMITS[0])


def add_axes_inches(fig, left, top, width, height):
    """Axes positioned in inches from the figure's top-left corner."""

    fig_width, fig_height = fig.get_size_inches()

    return fig.add_axes([left / fig_width, (fig_height - top - height) / fig_height,
                         width / fig_width, height / fig_height])


# ------------------------------------------------------------
# Layout: 2 x 3 slots - five maps and a legend panel
# ------------------------------------------------------------

FIG_W, LEFT, PANEL_W, GUTTER = 6.5, 0.45, 1.7, 0.15

PANEL_H = PANEL_W * MAP_ASPECT

HEADING, TITLE_BAND, ROW_GAP, BOTTOM = 0.26, 0.34, 0.2, 0.24

FIG_H = 0.05 + HEADING + 2 * (TITLE_BAND + PANEL_H) + ROW_GAP + BOTTOM

fig = plt.figure(figsize=(FIG_W, FIG_H))

fig.text(0.02 / FIG_W, 1 - 0.05 / FIG_H,
         f"RMSE of the proposed model minus each benchmark, per grid cell, "
         f"{FIRST_YEAR}–{LAST_YEAR}",
         fontsize=8, fontweight="bold", color=INK_2, va="top", ha="left")

slots = [
    (LEFT + column * (PANEL_W + GUTTER),
     0.05 + HEADING + row * (TITLE_BAND + PANEL_H + ROW_GAP) + TITLE_BAND)
    for row in range(2) for column in range(3)
]

for index, benchmark in enumerate(BENCHMARKS):

    row, column = divmod(index, 3)

    left, top = slots[index]

    ax = add_axes_inches(fig, left, top, PANEL_W, PANEL_H)

    values = grid.merge(differences[differences["model_B"] == benchmark],
                        on="grid_number", how="left")

    values.plot(ax=ax, column="RMSE_difference", cmap=CMAP, norm=NORM,
                edgecolor="white", linewidth=0.5, zorder=1)

    boundary.plot(ax=ax, facecolor="none", edgecolor=INK_2, linewidth=0.3, zorder=2)

    significant = values[values["significant_fdr"]]

    ax.scatter(significant["x"], significant["y"], s=9, color=INK,
               edgecolors="white", linewidths=0.5, zorder=4)

    ax.set_xlim(*LON_LIMITS)
    ax.set_ylim(*LAT_LIMITS)
    ax.set_aspect("equal")
    ax.set_xlabel("")
    ax.set_ylabel("")
    ax.set_xticks([118, 122, 126])
    ax.set_yticks([6, 10, 14, 18, 22])
    ax.set_xticklabels([f"{t}°E" for t in [118, 122, 126]])
    ax.set_yticklabels([f"{t}°N" for t in [6, 10, 14, 18, 22]])
    ax.tick_params(labelsize=6.5, labelbottom=row == 1 or index == 2, labelleft=column == 0)

    for side in ax.spines.values():
        side.set_color(AXIS)
        side.set_linewidth(0.6)

    lower = int((values["RMSE_difference"] < 0).sum())

    n_significant = int(values["significant_fdr"].sum())

    ax.text(0.0, 1.0 + 0.2 / PANEL_H, f"({'abcde'[index]}) vs {benchmark}",
            transform=ax.transAxes, fontsize=7.6, fontweight="bold", color=INK, ha="left", va="bottom")

    ax.text(0.0, 1.0 + 0.04 / PANEL_H,
            f"Lower RMSE in {lower}/{len(values)} · significant in {n_significant}",
            transform=ax.transAxes, fontsize=6.6, color=INK_2, ha="left", va="bottom")


# ------------------------------------------------------------
# Legend panel (sixth slot)
# ------------------------------------------------------------

left, top = slots[5]

fig.text((left + 0.05) / FIG_W, 1 - (top + 0.25) / FIG_H, "How to read",
         fontsize=7.6, fontweight="bold", color=INK, va="top", ha="left")

fig.text((left + 0.05) / FIG_W, 1 - (top + 0.5) / FIG_H,
         "ΔRMSE = RMSE of the proposed model\n"
         "minus RMSE of the benchmark, over\n"
         "the 120 months of each grid cell",
         fontsize=6.6, color=INK_2, va="top", ha="left", linespacing=1.3)

colorbar_ax = add_axes_inches(fig, left + 0.12, top + 1.12, PANEL_W - 0.17, 0.11)

colorbar = fig.colorbar(ScalarMappable(norm=NORM, cmap=CMAP), cax=colorbar_ax,
                        orientation="horizontal", extend="min",
                        ticks=[-0.08, -0.04, 0, 0.04, 0.08])

colorbar.ax.set_xticklabels(["−0.08", "−0.04", "0", "0.04", "0.08"])

colorbar.ax.tick_params(labelsize=6.5, length=2)

colorbar.outline.set_visible(False)

colorbar.set_label(f"ΔRMSE ({UNITS_LABEL})", fontsize=6.8, color=INK_2, labelpad=3)

fig.text((left + 0.05) / FIG_W, 1 - (top + 1.66) / FIG_H, "← proposed better",
         fontsize=6.4, color=INK_2, va="top", ha="left")

fig.text((left + PANEL_W - 0.05) / FIG_W, 1 - (top + 1.8) / FIG_H, "benchmark better →",
         fontsize=6.4, color=INK_2, va="top", ha="right")

marker_ax = add_axes_inches(fig, left + 0.05, top + 2.1, 0.12, 0.12)

marker_ax.scatter([0.5], [0.5], s=9, color=INK, edgecolors="white", linewidths=0.5)

marker_ax.set_xlim(0, 1)
marker_ax.set_ylim(0, 1)
marker_ax.axis("off")

fig.text((left + 0.22) / FIG_W, 1 - (top + 2.07) / FIG_H,
         "Significant difference\n(DM-HLN, FDR q < 0.05)",
         fontsize=6.6, color=INK_2, va="top", ha="left", linespacing=1.3)

fig.text((left + 0.05) / FIG_W, 1 - (top + 2.5) / FIG_H,
         "Differences below −0.08 take the\ndarkest blue (seasonal naive).",
         fontsize=6.4, color=INK_2, va="top", ha="left", linespacing=1.3, style="italic")

for extension in ("png", "svg", "pdf"):
    fig.savefig(FIGURE_DIR / f"rmse_difference_maps_full_record.{extension}",
                dpi=300, bbox_inches="tight", pad_inches=0.04, facecolor="white")

print(f"Saved: {FIGURE_DIR / 'rmse_difference_maps_full_record'}.png / .svg / .pdf")

plt.show()

## References

The test was written for this study using SciPy (Student's t distribution) and statsmodels (`multipletests` for the Holm and Benjamini-Hochberg adjustments).

- Diebold, F. X., & Mariano, R. S. (1995). Comparing predictive accuracy. *Journal of Business & Economic Statistics, 13*(3), 253–263. https://doi.org/10.1080/07350015.1995.10524599
- Harvey, D., Leybourne, S., & Newbold, P. (1997). Testing the equality of prediction mean squared errors. *International Journal of Forecasting, 13*(2), 281–291. https://doi.org/10.1016/S0169-2070(96)00719-4
- Diebold, F. X. (2015). Comparing predictive accuracy, twenty years later: A personal perspective on the use and abuse of Diebold–Mariano tests. *Journal of Business & Economic Statistics, 33*(1), 1–9. https://doi.org/10.1080/07350015.2014.983236
- Holm, S. (1979). A simple sequentially rejective multiple test procedure. *Scandinavian Journal of Statistics, 6*(2), 65–70.
- Benjamini, Y., & Hochberg, Y. (1995). Controlling the false discovery rate: A practical and powerful approach to multiple testing. *Journal of the Royal Statistical Society: Series B, 57*(1), 289–300. https://doi.org/10.1111/j.2517-6161.1995.tb02031.x
- Wilks, D. S. (2016). "The stippling shows statistically significant grid points": How research results are routinely overstated and overinterpreted, and what to do about it. *Bulletin of the American Meteorological Society, 97*(12), 2263–2273. https://doi.org/10.1175/BAMS-D-15-00267.1
- Virtanen, P., et al. (2020). SciPy 1.0: Fundamental algorithms for scientific computing in Python. *Nature Methods, 17*, 261–272. https://doi.org/10.1038/s41592-019-0686-2
- Seabold, S., & Perktold, J. (2010). Statsmodels: Econometric and statistical modeling with Python. *Proceedings of the 9th Python in Science Conference*, 92–96. https://doi.org/10.25080/Majora-92bf1922-011